In [10]:
from __future__ import annotations
import re
from pathlib import Path

import pandas as pd

## [1] 데이터 불러오기

In [11]:
path = Path(r"\\DocuONE\MyDrive\개인함\계량기_202604.xlsx")
df = pd.read_excel(path, sheet_name='자원리스트')
df

,11570,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 38,Unnamed: 39,Unnamed: 40,Unnamed: 41,Unnamed: 42,Unnamed: 43,Unnamed: 44,Unnamed: 45,Unnamed: 46,Unnamed: 47
0,고객센터,시/구/군,건물번호,단지번호,세대번호,세대유형,세대유형텍스트,평수,설치번호,설치유형,...,계량기유형명,설치번호생성일,동,전입일자,전출일자,특정구분,인덕션여부,설치,번,호생성일
1,H051,계룡시,1083314,H2500577,1048745,110,아파트,18,1039451,2,...,일반원격-원격연결 무,20020821,엄사면,20150614,99991231,비특정,연소기,2002,8,21
2,H051,계룡시,1083314,H2500577,1048751,110,아파트,18,1039467,2,...,일반원격-원격연결 무,20020821,엄사면,20140920,20260322,비특정,인덕션,2002,8,21
3,H051,계룡시,1083314,H2500577,1048760,110,아파트,18,1039477,2,...,일반원격-원격연결 무,20020821,엄사면,20151204,99991231,비특정,인덕션,2002,8,21
4,H051,계룡시,1083314,H2500577,1048771,110,아파트,18,1039491,2,...,일반원격-원격연결 무,20020821,엄사면,20061124,99991231,비특정,인덕션,2002,8,21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
842734,H075,유성구,4945206,H2010457,4420392,110,아파트,NaN,4489386,2,...,다기능-중앙집중,20191202,반석동,20240208,99991231,비특정,연소기,2019,12,2
842735,H075,유성구,4945206,H2010457,4420394,110,아파트,NaN,4489388,2,...,다기능-중앙집중,20191202,반석동,20200427,99991231,비특정,연소기,2019,12,2
842736,H075,유성구,4945206,H2010457,4420396,110,아파트,NaN,4489390,2,...,다기능-중앙집중,20191202,반석동,20240111,99991231,비특정,인덕션,2019,12,2
842737,H075,유성구,4945206,H2010457,4420398,110,아파트,NaN,4489392,2,...,다기능-중앙집중,20191202,반석동,20200427,99991231,비특정,인덕션,2019,12,2


In [12]:
df.dtypes

11570          object
Unnamed: 1     object
Unnamed: 2     object
Unnamed: 3     object
Unnamed: 4     object
Unnamed: 5     object
Unnamed: 6     object
Unnamed: 7     object
Unnamed: 8     object
Unnamed: 9     object
Unnamed: 10    object
Unnamed: 11    object
Unnamed: 12    object
Unnamed: 13    object
Unnamed: 14    object
Unnamed: 15    object
Unnamed: 16    object
Unnamed: 17    object
Unnamed: 18    object
Unnamed: 19    object
Unnamed: 20    object
Unnamed: 21    object
Unnamed: 22    object
Unnamed: 23    object
Unnamed: 24    object
Unnamed: 25    object
Unnamed: 26    object
Unnamed: 27    object
Unnamed: 28    object
Unnamed: 29    object
Unnamed: 30    object
Unnamed: 31    object
Unnamed: 32    object
Unnamed: 33    object
Unnamed: 34    object
Unnamed: 35    object
Unnamed: 36    object
Unnamed: 37    object
Unnamed: 38    object
Unnamed: 39    object
Unnamed: 40    object
Unnamed: 41    object
Unnamed: 42    object
Unnamed: 43    object
Unnamed: 44    object
Unnamed: 4

## [2] 조건 필터링

### (1) 최종인증년도 22~25년도

In [13]:
# 숫자로 변환 후 2022~2025만 필터
year = pd.to_numeric(df["최종인증년도"], errors="coerce")
cd1 = df[year.between(2022, 2025)]
cd1

KeyError: '최종인증년도'

In [ ]:
df['등급'].value_counts()

등급
4M           580948
2.5M         159595
6M            78673
10M           10111
16M            4638
25M            2918
40M            1516
65R             893
100R            736
3M              596
5M              431
160R            394
250T            220
2M              193
400T            192
65M             178
650T            139
7M              116
1000T            54
160T             53
1600T            51
250R             37
100T             31
100M              7
4000T             6
400R              4
6500T             3
650R              2
2500T             2
Corrector         1
Name: count, dtype: int64

In [ ]:
cd1['최종인증년도'].value_counts()

최종인증년도
2022    157116
2025    139465
2023    139448
2024    137813
Name: count, dtype: int64

In [ ]:
meter = df['계량기번호'].astype(str)

In [ ]:
cd1['등급'].value_counts()

등급
4M       410208
2.5M      94184
6M        57369
10M        6436
16M        2245
25M        1394
40M         650
65R         422
100R        339
160R        181
250T         97
400T         85
650T         69
65M          66
1600T        29
1000T        26
250R         16
160T         11
100T          8
650R          2
2500T         2
4000T         2
400R          1
Name: count, dtype: int64

### (2) 소용량 계량기만 분류
- 2, 2.5M = G1.6
- 3, 4M = G2.5
- 5, 6, 7M = G4

In [ ]:
cd2 = cd1.loc[
    cd1["등급"].astype("string").isin(["2M", "2.5M", "3M", "4M", "5M", "6M", "7M"]),
]
cd2[['최종인증년도','계량기번호','등급']].sort_values(by='최종인증년도', ascending=False)

,최종인증년도,계량기번호,등급
95596,2025,HM199422004682,6M
95597,2025,HM259322089318,4M
95598,2025,HM259322089316,4M
486385,2025,HM253214515913,2.5M
514224,2025,HM259212000208,2.5M
...,...,...,...
12,2022,HM169412005513,6M
11,2022,HM179412000953,6M
10,2022,HM179422002030,6M
9,2022,HM159422016620,6M


In [ ]:
cd2['등급'].value_counts()

등급
4M      410208
2.5M     94184
6M       57369
Name: count, dtype: int64

### (3) 현재 사용량 분리
- 전출일자 = 9999.12.31

In [ ]:
cd3 = cd2[cd2['전출일자'] == 99991231]
cd3

,고객센터,시/구/군,건물번호,단지번호,세대번호,세대유형,세대유형텍스트,평수,설치번호,설치유형,...,계량기유형명,설치번호생성일,동,전입일자,전출일자,특정구분,인덕션여부,설치,번,호생성일
0,H051,계룡시,1083314,H2500577,1048745,110,아파트,18,1039451,2,...,일반원격-원격연결 무,20020821,엄사면,20150614,99991231,비특정,연소기,2002,8,21
2,H051,계룡시,1083314,H2500577,1048760,110,아파트,18,1039477,2,...,일반원격-원격연결 무,20020821,엄사면,20151204,99991231,비특정,인덕션,2002,8,21
3,H051,계룡시,1083314,H2500577,1048771,110,아파트,18,1039491,2,...,일반원격-원격연결 무,20020821,엄사면,20061124,99991231,비특정,인덕션,2002,8,21
4,H051,계룡시,1083314,H2500577,1048796,110,아파트,18,1039503,2,...,일반원격-원격연결 무,20020821,엄사면,20020816,99991231,비특정,연소기,2002,8,21
5,H051,계룡시,1083314,H2500577,1048847,110,아파트,18,1039512,2,...,일반원격-원격연결 무,20020821,엄사면,20240822,99991231,비특정,인덕션,2002,8,21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
842733,H075,유성구,4945206,H2010457,4420392,110,아파트,NaN,4489386,2,...,다기능-중앙집중,20191202,반석동,20240208,99991231,비특정,연소기,2019,12,2
842734,H075,유성구,4945206,H2010457,4420394,110,아파트,NaN,4489388,2,...,다기능-중앙집중,20191202,반석동,20200427,99991231,비특정,연소기,2019,12,2
842735,H075,유성구,4945206,H2010457,4420396,110,아파트,NaN,4489390,2,...,다기능-중앙집중,20191202,반석동,20240111,99991231,비특정,인덕션,2019,12,2
842736,H075,유성구,4945206,H2010457,4420398,110,아파트,NaN,4489392,2,...,다기능-중앙집중,20191202,반석동,20200427,99991231,비특정,인덕션,2019,12,2


In [ ]:
cd3['등급'].value_counts()

등급
4M      386886
2.5M     79362
6M       56067
Name: count, dtype: int64

In [ ]:
cd3['신품/검정품'].value_counts()

신품/검정품
1.0    306881
2.0    215434
Name: count, dtype: int64

## [3] 집계 테이블 만들기

In [ ]:
tmp = cd3[["최종인증년도", "신품/검정품", "등급"]].copy()

### (1) 등급 코드 생성

In [ ]:
grade_map = {
    "2M": "G1.6",
    "2.5M": "G1.6",
    "3M": "G2.5",
    "4M": "G2.5",
    "5M": "G4",
    "6M": "G4",
    "7M": "G4"
}


In [ ]:
tmp["등급코드"] = tmp["등급"].astype("string").str.strip().map(grade_map)

In [ ]:
tmp["등급표시"] = tmp["등급"] + "(" + tmp["등급코드"] + ")"

In [ ]:
tmp["신품/검정품"] = tmp["신품/검정품"].replace({
    1: "신품", 2: "검정",
    "1": "신품", "2": "검정"
})
tmp

,최종인증년도,신품/검정품,등급,등급코드,등급표시
0,2022,검정,6M,G4,6M(G4)
2,2022,검정,6M,G4,6M(G4)
3,2022,검정,6M,G4,6M(G4)
4,2022,검정,6M,G4,6M(G4)
5,2022,검정,6M,G4,6M(G4)
...,...,...,...,...,...
842733,2024,신품,4M,G2.5,4M(G2.5)
842734,2024,신품,4M,G2.5,4M(G2.5)
842735,2024,신품,4M,G2.5,4M(G2.5)
842736,2024,신품,4M,G2.5,4M(G2.5)


In [ ]:
tmp.isnull().sum()

최종인증년도    0
신품/검정품    0
등급        0
등급코드      0
dtype: int64

### (2) 집계

In [ ]:
result = pd.pivot_table(
    tmp,
    index="최종인증년도",
    columns=["신품/검정품", "등급표시"],
    values="등급",
    aggfunc="count",
    fill_value=0
)
result

신품/검정품         검정                         신품                
등급표시   2.5M(G1.6) 4M(G2.5) 6M(G4) 2.5M(G1.6) 4M(G2.5) 6M(G4)
최종인증년도                                                      
2022        10932    38262   8231      11106    67281   6428
2023        11820    39415   5329       6040    59181   5874
2024         7937    31891   5831      11847    58100  10416
2025         5789    43618   6379      13891    49138   7579

In [ ]:
# result.to_excel("소용량 계량기 필요량 집계.xlsx", sheet_name='집계표')